In [1]:
%pip install pgembed psycopg2-binary sqlalchemy pandas kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 50.1 MB/s eta 0:00:00


In [2]:
%pip install -q psycopg2-binary psycopg pg8000

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.9 MB/s eta 0:00:00


In [3]:
import os
import pandas as pd
import pgembed as pgserver
from sqlalchemy import create_engine, text
import kagglehub

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

# Варіант A: якщо Kaggle credentials налаштовані
dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print("Dataset directory:", dataset_dir)

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


PostgreSQL 17.9 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 14.2.1 20250110 (Red Hat 14.2.1-11), 64-bit
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset directory: /kaggle/input/brazilian-ecommerce


In [4]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [5]:
create_schema_query = text("""
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""")

with engine.begin() as conn:
    conn.execute(create_schema_query)
print("Typed schema created successfully!")

Typed schema created successfully!


In [6]:
insert_data_query = text("""
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;
""")

with engine.begin() as conn:
    conn.execute(insert_data_query)

smoke_test_query = """
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;
"""
print(pd.read_sql(smoke_test_query, engine))

    table_name  n_rows
0    customers   99441
1  order_items  112650
2       orders   99441
3     products   32951
4      sellers    3095


In [7]:
query_2 = text("""
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;
""")

with engine.begin() as conn:
    result = pd.read_sql(query_2, conn)
print(result)

                         customer_id customer_state  recency_days  frequency  \
0   1617b1357756262bfa56ab541c47bc16             RJ           384          1   
1   ec5b2ba62e574342386871631fafd3fc             ES            95          1   
2   c6e2731c5b391845f6800c97401a43a9             MS           613          1   
3   f48d464a0baaea338cb25f816991ab1f             ES            85          1   
4   3fd6777bbce08a352fddd04e4a7cc8f6             SP           512          1   
5   05455dfa7cd02f13d132aa7a6a9729c6             MG           328          1   
6   df55c14d1476a9a3467f131269c2477f             RJ           565          1   
7   e0a2412720e9ea4f26c1ac985f6a7358             GO            98          1   
8   24bbf5fd2f2e1b359ee7de94defc4a15             SP           548          1   
9   3d979689f636322c62418b6346b1c6d2             PB           118          1   
10  1afc82cd60e303ef09b4ef9837c9505c             SP            76          1   
11  cc803a2c412833101651d3f90ca7de24    

In [8]:
from sqlalchemy import text
import pandas as pd

# 1. Створення та наповнення таблиці hw5_referrals
setup_referrals_query = text("""
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""")

# Виконуємо DDL/DML запит
with engine.begin() as conn:
    conn.execute(setup_referrals_query)
print("Синтетичну таблицю hw5_referrals створено та наповнено!\n")

# 2. Виведення результату створення таблиці
check_referrals_query = text("""
SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;
""")
print("Вміст hw5_referrals:")
print(pd.read_sql(check_referrals_query, engine))
print("\n" + "-"*50 + "\n")

# 3. Рекурсивний CTE для обчислення depth і path
recursive_cte_query = text("""
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""")

# Виконання та виведення результатів рекурсивного запиту
print("Результати Recursive CTE:")
print(pd.read_sql(recursive_cte_query, engine))

Синтетичну таблицю hw5_referrals створено та наповнено!

Вміст hw5_referrals:
               referrer_customer_id              referred_customer_id  \
0  00012a2ce6f8dcda20d059ce98491703  000161a058600d5901f007fab4c27140   
1  000161a058600d5901f007fab4c27140  0001fd6190edaaf884bcaf3d49edf079   
2  0001fd6190edaaf884bcaf3d49edf079  0002414f95344307404f0ace7a26f1d5   
3  0002414f95344307404f0ace7a26f1d5  000379cdec625522490c315e70c7a9fb   
4  000379cdec625522490c315e70c7a9fb  0004164d20a9e969af783496f3408652   
5  0004164d20a9e969af783496f3408652  000419c5494106c306a97b5635748086   
6  000419c5494106c306a97b5635748086  00046a560d407e99b969756e0b10f282   
7  00046a560d407e99b969756e0b10f282  00050bf6e01e69d5c0fd612f1bcfb69c   
8  00050bf6e01e69d5c0fd612f1bcfb69c  000598caf2ef4117407665ac33275130   

                       referred_at  
0 2026-10-06 15:13:02.088651+00:00  
1 2026-10-06 15:13:02.088651+00:00  
2 2026-10-06 15:13:02.088651+00:00  
3 2026-10-06 15:13:02.088651+00:00  
4 2026

In [9]:
from sqlalchemy import text
import pandas as pd

# 1. Створення Materialized View та унікального індексу
create_mv_query = text("""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
""")

with engine.begin() as conn:
    conn.execute(create_mv_query)
print("Materialized View та UNIQUE INDEX успішно створено!\n")


# 2. Перевірка MV
print("Перевірка кількості рядків:")
count_query = text("SELECT COUNT(*) AS n_rows FROM mv_olist_customer_features;")
print(pd.read_sql(count_query, engine))

print("\nТоп-5 клієнтів з MV:")
sample_query = text("""
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
""")
print(pd.read_sql(sample_query, engine))
print("\n" + "-"*50 + "\n")


# 3. Виконання CONCURRENT REFRESH
# Важливо: AUTOCOMMIT потрібен для REFRESH CONCURRENTLY в SQLAlchemy
refresh_query = text("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;")

with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(refresh_query)
print("REFRESH MATERIALIZED VIEW CONCURRENTLY виконано успішно!")

Materialized View та UNIQUE INDEX успішно створено!

Перевірка кількості рядків:
   n_rows
0   99441

Топ-5 клієнтів з MV:
                        customer_id customer_state  recency_days  frequency  \
0  1617b1357756262bfa56ab541c47bc16             RJ           384          1   
1  ec5b2ba62e574342386871631fafd3fc             ES            95          1   
2  c6e2731c5b391845f6800c97401a43a9             MS           613          1   
3  f48d464a0baaea338cb25f816991ab1f             ES            85          1   
4  3fd6777bbce08a352fddd04e4a7cc8f6             SP           512          1   

   monetary first_order_date cohort_quarter  avg_order_value_30d_recent  \
0  13664.08       2017-09-29     2017-07-01                    13664.08   
1   7274.88       2018-07-15     2018-07-01                     7274.88   
2   6929.31       2017-02-12     2017-01-01                     6929.31   
3   6922.21       2018-07-25     2018-07-01                     6922.21   
4   6726.66       2017-05-2

In [10]:
from sqlalchemy import text

def print_explain_plan(engine, query, title):
    print(f"\n{'='*10} {title} {'='*10}")
    with engine.connect() as conn:
        result = conn.execute(query)
        for row in result:
            print(row[0])

# 1. EXPLAIN на повному CTE-pipeline
explain_cte_query = text("""
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

# 2. EXPLAIN на lookup із MV
explain_mv_lookup_query = text("""
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id;
""")

# 3. Створення індексу та EXPLAIN на top-N
create_idx_query = text("""
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);
""")

explain_mv_topn_query = text("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;
""")

# Виконання
print_explain_plan(engine, explain_cte_query, "EXPLAIN: Повний CTE-pipeline")
print_explain_plan(engine, explain_mv_lookup_query, "EXPLAIN: Lookup із Materialized View")

with engine.begin() as conn:
    conn.execute(create_idx_query)
print("\nІндекс idx_mv_olist_customer_features_monetary створено.")

print_explain_plan(engine, explain_mv_topn_query, "EXPLAIN: Top-N з Materialized View")


========== EXPLAIN: Повний CTE-pipeline ==========
Limit  (cost=34400.85..34400.88 rows=10 width=64) (actual time=991.225..994.773 rows=10 loops=1)
  Buffers: shared hit=5966, temp read=3209 written=3947
  ->  Sort  (cost=34400.85..34401.35 rows=200 width=64) (actual time=991.223..994.769 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=5966, temp read=3209 written=3947
        ->  Subquery Scan on stage2_rfm  (cost=34391.03..34396.53 rows=200 width=64) (actual time=802.134..959.365 rows=99441 loops=1)
              Buffers: shared hit=5966, temp read=3209 written=3947
              ->  HashAggregate  (cost=34391.03..34394.53 rows=200 width=68) (actual time=802.132..946.068 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                 

# Аналіз продуктивності та архітектурних рішень Feature Store

Аналіз результатів EXPLAIN ANALYZE чітко демонструє різницю між агрегацією на льоту та використанням матеріалізованого кешу для ML-ознак.  



*   Найдорожчі операції: У повному CTE-pipeline (загальний час виконання ~998 мс) найдорожчими є операції групування. Вузол HashAggregate забирає левову частку часу (від 802 до 946 мс). Під капотом йому передує Finalize GroupAggregate, який спирається на надзвичайно важкий Sort. У плані видно Sort Method: external sort, що означає нестачу оперативної пам'яті (work_mem) — база змушена скидати тимчасові файли на диск (Disk: 5432kB, temp read=3209 written=3947). Також ресурсомістким є з'єднання Parallel Hash Right Join.

*   Доцільність MV та Refresh-cycle: Кешувати цей pipeline через Materialized View однозначно варто. При lookup-запиті до MV час виконання падає з майже секунди до 0.099 мс завдяки використанню Index Scan using idx_mv_olist_customer_features_customer_id. Оптимальний refresh-cycle для RFM-метрик — щодоби (вночі), оскільки показники LTV чи частоти покупок не змінюються радикально щохвилини, а їх переобчислення сильно навантажує I/O.

*   VIEW vs Materialized View: Звичайний VIEW був би доречним у транзакційних OLTP-сценаріях (наприклад, перевірка поточного статусу замовлення клієнта перед білінгом). Там критична 100% свіжість даних, а точковий запит WHERE customer_id = X виконає швидкий Index Scan по базовій таблиці без необхідності сканувати весь датасет.

*   Захист Recursive CTE: Термінація гарантується умовами NOT ... = ANY(path_ids) та лімітом depth < 50. Якби в referral-графі був цикл A → B → A, без масиву відвіданих вузлів рекурсія стала б нескінченною, що призвело б до падіння бази через вичерпання пам'яті (OOM).

*   Головний trade-off: Архітектура цього pipeline ставить у пріоритет наднизьку затримку (latency) та економію процесорного часу (recompute-cost). Ми свідомо жертвуємо абсолютною актуальністю (freshness) і погоджуємося на додаткові витрати пам'яті для кешу та індексів (storage-cost), щоб уникнути дорогих дискових сортувань при кожному зверненні ML-моделі до Feature Store.

